In [1]:
import torch
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from torch_geometric.loader import DataLoader

print("Libraries loaded.")

/Users/Mourya/miniforge3/envs/aircraft-conflict-gnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.


In [2]:
device = torch.device("cpu")

print(device)

cpu


In [3]:
test_graphs = torch.load(
    "processed_data/test_graphs.pt",
    weights_only=False
)

print(len(test_graphs))

108


In [4]:
test_loader = DataLoader(
    test_graphs,
    batch_size=8,
    shuffle=False
)

print("Test loader ready.")

Test loader ready.


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GATConv


class ConflictGAT(nn.Module):

    def __init__(self,
                 node_dim=6,
                 edge_dim=5,
                 hidden_dim=64):

        super().__init__()

        self.gat1 = GATConv(
            node_dim,
            hidden_dim,
            heads=2,
            concat=True,
            edge_dim=edge_dim
        )

        self.gat2 = GATConv(
            hidden_dim * 2,
            hidden_dim,
            heads=1,
            concat=False,
            edge_dim=edge_dim
        )

        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, data):

        x = F.elu(
            self.gat1(
                data.x,
                data.edge_index,
                data.edge_attr
            )
        )

        x = self.gat2(
            x,
            data.edge_index,
            data.edge_attr
        )

        src = x[data.edge_index[0]]
        dst = x[data.edge_index[1]]

        edge_input = torch.cat(
            [
                src,
                dst,
                data.edge_attr
            ],
            dim=1
        )

        return self.edge_mlp(edge_input).squeeze(-1)


gat_model = ConflictGAT()

print(gat_model)

ConflictGAT(
  (gat1): GATConv(6, 64, heads=2)
  (gat2): GATConv(128, 64, heads=1)
  (edge_mlp): Sequential(
    (0): Linear(in_features=133, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [6]:
state = torch.load(
    "models/conflict_gat_large_best.pth",
    map_location="cpu",
    weights_only=False
)

gat_model.load_state_dict(state)

gat_model.eval()

print("Final GAT loaded successfully.")

Final GAT loaded successfully.


In [7]:
all_labels = []
all_probs = []

with torch.no_grad():

    for batch in test_loader:

        batch = batch.to(device)

        logits = gat_model(batch)

        probs = torch.sigmoid(logits)

        all_labels.extend(
            batch.y.cpu().numpy()
        )

        all_probs.extend(
            probs.cpu().numpy()
        )

print(len(all_labels))
print(len(all_probs))

130899
130899


In [8]:
import numpy as np
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

labels = np.array(all_labels)
probs = np.array(all_probs)

predictions = (probs >= 0.5).astype(int)

print("Precision:",
      precision_score(labels, predictions, zero_division=0))

print("Recall:",
      recall_score(labels, predictions, zero_division=0))

print("F1:",
      f1_score(labels, predictions, zero_division=0))

print("ROC-AUC:",
      roc_auc_score(labels, probs))

Precision: 0.004775549188156638
Recall: 0.3333333333333333
F1: 0.009416195856873822
ROC-AUC: 0.9419523849549294


In [9]:
pred_df = pd.DataFrame({

    "label": labels,

    "probability": probs,

    "prediction": predictions

})

pred_df.to_csv(
    "processed_data/final_gat_predictions.csv",
    index=False
)

print(pred_df.head())

print(pred_df.shape)

   label   probability  prediction
0      0  2.002922e-11           0
1      0  3.420755e-09           0
2      0  1.391872e-17           0
3      0  0.000000e+00           0
4      0  3.834392e-27           0
(130899, 3)


In [10]:
metrics = pd.DataFrame({

    "Model":["GAT"],

    "Precision":[precision_score(labels,predictions,zero_division=0)],

    "Recall":[recall_score(labels,predictions,zero_division=0)],

    "F1":[f1_score(labels,predictions,zero_division=0)],

    "ROC_AUC":[roc_auc_score(labels,probs)]

})

metrics.to_csv(
    "processed_data/final_gat_metrics.csv",
    index=False
)

metrics

,Model,Precision,Recall,F1,ROC_AUC
0,GAT,0.004776,0.333333,0.009416,0.941952
